<h1 style="text-align: center;"> Data Preprocessing & Feature Engineering </h1>
<h4 style="text-align: center;">  From Raw Data to Model-Ready Features </h4>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import seaborn as sns
import plotly.express as px

# Set Float precision
pd.options.display.float_format = '{:,.3f}'.format
# Set visual style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


In [2]:
df = pd.read_csv('../data/cleaned_data.csv')
df.head()

,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,Price,Brand,Model
0,Mumbai,2010,72000,CNG,Manual,First,11.438,998.000,58.160,5.000,1.750,Maruti,Wagon
1,Pune,2015,41000,Diesel,Manual,First,19.670,"1,582.000",126.200,5.000,12.500,Hyundai,Creta
2,Chennai,2011,46000,Petrol,Manual,First,18.200,"1,199.000",88.700,5.000,4.500,Honda,Jazz
3,Chennai,2012,87000,Diesel,Manual,First,20.770,"1,248.000",88.760,7.000,6.000,Maruti,Ertiga
4,Coimbatore,2013,40670,Diesel,Automatic,Second,15.200,"1,968.000",140.800,5.000,17.740,Audi,A4


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6015 entries, 0 to 6014
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Location           6015 non-null   object 
 1   Year               6015 non-null   int64  
 2   Kilometers_Driven  6015 non-null   int64  
 3   Fuel_Type          6015 non-null   object 
 4   Transmission       6015 non-null   object 
 5   Owner_Type         6015 non-null   object 
 6   Mileage            5945 non-null   float64
 7   Engine             5979 non-null   float64
 8   Power              5873 non-null   float64
 9   Seats              5973 non-null   float64
 10  Price              6015 non-null   float64
 11  Brand              6015 non-null   object 
 12  Model              6015 non-null   object 
dtypes: float64(5), int64(2), object(6)
memory usage: 611.0+ KB


In [4]:
df['Seats'] = df['Seats'].astype('category')

In [5]:
df.describe()

,Year,Kilometers_Driven,Mileage,Engine,Power,Price
count,"6,015.000","6,015.000","5,945.000","5,979.000","5,873.000","6,015.000"
mean,"2,013.358","57,670.792",18.195,"1,619.955",113.128,9.425
std,3.270,"37,870.190",4.152,598.896,53.507,10.905
min,"1,998.000",171.000,5.676,72.000,34.200,0.440
25%,"2,011.000","34,000.000",15.150,"1,198.000",75.000,3.500
50%,"2,014.000","53,000.000",18.120,"1,493.000",97.700,5.630
75%,"2,016.000","73,000.000",21.030,"1,984.000",138.100,9.950
max,"2,019.000","775,000.000",28.400,"5,998.000",552.000,100.000


In [6]:
# First, Split the data into train and test sets
from sklearn.model_selection import train_test_split

X = df.drop(['Price'] , axis=1)
y = df['Price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
num_features = X.select_dtypes(include=[np.number]).columns.to_list()
num_features

['Year', 'Kilometers_Driven', 'Mileage', 'Engine', 'Power']

In [8]:
cat_features = X.select_dtypes(include=['category', 'object']).columns.to_list()
cat_features

['Location',
 'Fuel_Type',
 'Transmission',
 'Owner_Type',
 'Seats',
 'Brand',
 'Model']

# Categorical Features Handling

### ✅ One-Hot Encoding

**Purpose:**
- Convert categorical variables into **binary columns** (0/1) for each category.
- Useful for **nominal features** with no inherent order.

**Mathematical Concept:**
Given a feature $color \in \{\text{Red, Green, Blue}\}$:

| color | Red | Green | Blue |
|-------|-----|-------|------|
| Red   | 1   | 0     | 0    |
| Green | 0   | 1     | 0    |
| Blue  | 0   | 0     | 1    |

**Notes:**
- Creates $k$ columns for $k$ categories.  
- Can cause **high dimensionality** if the feature has many categories.



In [9]:
# One-Hot Encoding using sklearn
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

df = pd.DataFrame({'Color': ['Red', 'Green', 'Blue', 'Red']})

ohe = OneHotEncoder(sparse_output=False, drop='first')  # drop='first' avoids dummy variable trap
df_ohe = pd.DataFrame(ohe.fit_transform(df), columns=ohe.get_feature_names_out(['Color']))
df_ohe


,Color_Green,Color_Red
0,0.000,1.000
1,1.000,0.000
2,0.000,0.000
3,0.000,1.000


### ✅ Ordinal Encoding

**Purpose:**
- Convert **ordered categorical variables** into integers reflecting order.
- Useful for features like `Low < Medium < High`.

**Mathematical Concept:**
If $size \in \{\text{Small, Medium, Large}\}$:

$$
\text{Small} \rightarrow 0, \quad \text{Medium} \rightarrow 1, \quad \text{Large} \rightarrow 2
$$

**Notes:**
- Preserves order information.  
- Not suitable for **nominal features**, because the model may interpret numbers as continuous.





In [10]:
# Ordinal Encoding using sklearn
from sklearn.preprocessing import OrdinalEncoder

df = pd.DataFrame({'Size': ['Small', 'Medium', 'Large', 'Medium']})

ord_enc = OrdinalEncoder(categories=[['Small', 'Medium', 'Large']])
df_ord = pd.DataFrame(ord_enc.fit_transform(df), columns=['Size'])
df_ord


,Size
0,0.000
1,1.000
2,2.000
3,1.000


### ✅ Frequency Encoding

**Purpose:**
- Encode categorical variables using **frequency of each category** in the dataset.
- Reduces dimensionality while keeping **information about occurrence**.

**Mathematical Concept:**
For feature $color$:

$$
\text{Frequency}(category) = \frac{\text{count(category)}}{\text{total samples}}
$$

Example:

| color | Count | Frequency |
|-------|-------|-----------|
| Red   | 2     | 2/4 = 0.5 |
| Green | 1     | 1/4 = 0.25 |
| Blue  | 1     | 1/4 = 0.25 |


In [11]:
# Frequency encoding
df = pd.DataFrame({'Color': ['Red', 'Green', 'Blue', 'Red']})
freq = df['Color'].value_counts(normalize=True)
df['Color_freq'] = df['Color'].map(freq)
df


,Color,Color_freq
0,Red,0.500
1,Green,0.250
2,Blue,0.250
3,Red,0.500


### ✅ Target Encoding

**Purpose:**
- Encode categorical features using **mean of the target variable** for each category.
- Captures correlation between categorical feature and target.

**Mathematical Concept:**
For a feature $color$ and target $y$:

$$
\text{Encoded}(color_i) = \frac{\sum_{j: color_j = color_i} y_j}{\text{count of } color_i}
$$

**Notes:**
- Can **leak information** if not applied carefully (use cross-validation or separate train/test).  
- Works well for high cardinality features.



In [12]:
# Target encoding
df = pd.DataFrame({'Color': ['Red', 'Green', 'Blue', 'Red'], 'Price':[100, 200, 150, 120]})
target_mean = df.groupby('Color')['Price'].mean()
df['Color_target'] = df['Color'].map(target_mean)
df


,Color,Price,Color_target
0,Red,100,110.000
1,Green,200,200.000
2,Blue,150,150.000
3,Red,120,110.000


### ✅ Binary Encoding

**Purpose:**
- Encode **high-cardinality categorical features** using a **binary representation**.
- Reduces dimensionality compared to one-hot encoding for features with many unique categories.

**Mathematical Concept:**
1. Assign each category a unique integer:  
   $$
   \text{Category}_i \rightarrow \text{Integer}_i
   $$
2. Convert the integer into **binary digits**:  
   $$
   \text{Integer}_i \rightarrow [b_0, b_1, ..., b_k]
   $$
   where $b_k$ is the $k$-th binary bit.
3. Each binary bit becomes a separate column in the dataset.

**Notes:**
- Useful for **high-cardinality features** like `Brand` or `Model`.  
- Output has fewer columns than one-hot encoding.  
- Can be combined with **other categorical encodings** (one-hot, ordinal) in a pipeline.


### ✅ Summary Table:

| Method     | Use Case                                         | Pros                                        | Cons                             |
| ---------- | ----------------------------------------------- | ------------------------------------------- | -------------------------------- |
| One-Hot    | Nominal                                         | Preserves information, no order assumption | High dimensionality              |
| Ordinal    | Ordered                                         | Preserves order                             | Model may assume linear distance |
| Frequency  | Nominal/High cardinality                        | Low dimensionality                          | No order, may lose detail        |
| Target     | Nominal/High cardinality, correlated with target | Captures predictive power                   | Risk of target leakage           |
| Binary     | High-cardinality categorical features           | Reduces dimensionality vs one-hot, preserves uniqueness | May be harder to interpret, numeric representation may be misleading |


In [13]:
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np
import pandas as pd

class OutlierHandler(BaseEstimator, TransformerMixin):
    def __init__(self, method='iqr', lower_percentile=25, upper_percentile=75,
                 iqr_factor=1.5, z_thresh=3.0, handle='winsorize'):
        self.method = method
        self.lower_percentile = lower_percentile
        self.upper_percentile = upper_percentile
        self.iqr_factor = iqr_factor
        self.z_thresh = z_thresh
        self.handle = handle
        self.bounds_ = {}
        self.columns_ = None

    def fit(self, X, y=None):
        # Ensure input is a DataFrame
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X)

        self.columns_ = X.select_dtypes(include=np.number).columns

        if self.method == 'iqr':
            Q1 = X[self.columns_].quantile(self.lower_percentile / 100.0)
            Q3 = X[self.columns_].quantile(self.upper_percentile / 100.0)
            IQR = Q3 - Q1
            lower = Q1 - self.iqr_factor * IQR
            upper = Q3 + self.iqr_factor * IQR
        elif self.method == 'zscore':
            mean = X[self.columns_].mean()
            std = X[self.columns_].std()
            lower = mean - self.z_thresh * std
            upper = mean + self.z_thresh * std
        else:
            raise ValueError("method must be 'iqr' or 'zscore'")

        self.bounds_ = {col: (lower[col], upper[col]) for col in self.columns_}
        return self

    def transform(self, X):
        # Convert to DataFrame (keep column names if possible)
        if isinstance(X, np.ndarray):
            if self.columns_ is not None and X.shape[1] == len(self.columns_):
                X = pd.DataFrame(X, columns=self.columns_)
            else:
                X = pd.DataFrame(X)

        X_copy = X.copy()

        # Apply outlier handling
        for col, (lower, upper) in self.bounds_.items():
            if self.handle == 'remove':
                X_copy[col] = X_copy[col].where((X_copy[col] >= lower) & (X_copy[col] <= upper))
            elif self.handle == 'winsorize':
                X_copy[col] = np.clip(X_copy[col], lower, upper)
            else:
                raise ValueError("handle must be 'remove' or 'winsorize'")
        
        return X_copy

    def fit_transform(self, X, y=None, **fit_params):
        return self.fit(X, y).transform(X)

    def get_feature_names_out(self, input_features=None):
        # Return the same feature names
        if input_features is None and hasattr(self, "columns_"):
            return self.columns_
        return input_features


In [14]:
# !pip install category-encoders

In [15]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler, OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from category_encoders import BinaryEncoder

# --- Column groups ---
numeric_cols = ['Kilometers_Driven', 'Mileage', 'Engine', 'Power', 'Seats']
year_cols = ['Year']
onehot_cols = ['Location', 'Fuel_Type', 'Transmission']  # low-cardinality
binary_cols = ['Brand', 'Model']  # high-cardinality
ordinal_cols = ['Owner_Type']
ordinal_categories = [['Fourth & Above', 'Third', 'Second', 'First']]

# --- Numeric pipeline ---
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('outlier', OutlierHandler(handle='winsorize')),
    ('scaler', MinMaxScaler())
])

# --- Year pipeline ---
year_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MinMaxScaler())
])

# --- Categorical pipelines ---
onehot_enc = OneHotEncoder(sparse_output=False, drop='first')
binary_enc = BinaryEncoder()
ordinal_enc = OrdinalEncoder(categories=ordinal_categories)

# --- Combine pipelines into ColumnTransformer ---
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_cols),
    ('year', year_pipeline, year_cols),
    ('onehot', onehot_enc, onehot_cols),
    ('binary', binary_enc, binary_cols),
    ('ordinal', ordinal_enc, ordinal_cols)
], remainder='passthrough', verbose_feature_names_out=False).set_output(transform='pandas')

# --- Fit and transform ---
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)


In [16]:
X_train_processed

,Kilometers_Driven,Mileage,Engine,Power,Seats,Year,Location_Bangalore,Location_Chennai,Location_Coimbatore,Location_Delhi,...,Brand_4,Model_0,Model_1,Model_2,Model_3,Model_4,Model_5,Model_6,Model_7,Owner_Type
5639,1.000,0.399,0.497,0.308,0.000,0.381,0.000,0.000,0.000,0.000,...,1,0,0,0,0,0,0,0,1,1.000
2303,0.845,0.468,0.467,0.339,0.000,0.381,0.000,1.000,0.000,0.000,...,0,0,0,0,0,0,0,1,0,3.000
2617,0.999,0.570,0.434,0.179,0.000,0.476,0.000,0.000,0.000,0.000,...,1,0,0,0,0,0,0,1,1,3.000
1397,0.275,0.425,0.957,1.000,0.000,0.857,0.000,0.000,0.000,0.000,...,0,0,0,0,0,0,1,0,0,3.000
3703,0.395,0.569,0.495,0.475,0.000,0.714,0.000,0.000,0.000,1.000,...,1,0,0,0,0,0,1,0,1,3.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3772,0.115,0.561,0.369,0.247,0.000,0.810,0.000,1.000,0.000,0.000,...,1,0,0,0,1,1,0,1,0,3.000
5191,0.356,0.618,0.467,0.282,0.000,0.762,0.000,0.000,0.000,0.000,...,1,0,0,1,1,1,0,1,1,2.000
5226,0.565,0.736,0.630,0.804,0.000,0.667,1.000,0.000,0.000,0.000,...,0,0,0,1,1,1,0,1,0,3.000
5390,0.350,0.561,0.369,0.246,0.000,0.571,0.000,0.000,0.000,0.000,...,1,0,0,0,1,1,0,1,0,3.000
